##### 🥇 GOLD LAYER

Cleansed, aggregated and business-ready data derived from Silver layer for reporting and analytics.

In [71]:
#Load environment variables from .env files 
from dotenv import load_dotenv
import os

load_dotenv("coding.env")

True

In [72]:
#Initiate Spark Session

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
sys.path.append("C:\\hadoop\\bin")

from pyspark.sql import SparkSession


from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("PyProject") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.1.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

In [73]:
#Reading Delta Lake from Silver layer to a new dataframe

silver_path = os.getenv("SILVER_PATH")
gold_delta_lake = spark.read.format("delta").load(silver_path)

In [74]:
gold_delta_lake.show(truncate=False)

+--------------+-----------+----------------------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------------+
|Transaction_ID|Customer_ID|Category                          |Item        |Price_Per_Unit|Quantity|Total_Spent|Payment_Method|Location|Transaction_Date|Discount_Applied|datetime_now              |
+--------------+-----------+----------------------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------------+
|TXN_5328604   |CUST_07    |Food                              |Item_20_FOOD|33.5          |10.0    |335.0      |Cash          |Online  |2024-07-08      |False           |2026-03-03 13:23:26.469992|
|TXN_2634868   |CUST_13    |Patisserie                        |Item_11_PAT |20.0          |4.0     |80.0       |Cash          |In-store|2024-06-26      |False           |2026-03-03 13:23:26.469992|
|TXN_47932

AGGREGATIONS  

In [75]:
# Calculating Total Revenue by Category

from pyspark.sql.functions import sum, count, desc

# Total revenue by category
df_revenue_by_category = gold_delta_lake.groupBy("Category").agg(
    sum("Total_Spent").alias("Total_Revenue"),
    sum("Quantity").alias("Total_Quantity_Sold"),
    count("Transaction_ID").alias("Total_Transactions")
)

df_revenue_by_category.show()

+--------------------+-------------+-------------------+------------------+
|            Category|Total_Revenue|Total_Quantity_Sold|Total_Transactions|
+--------------------+-------------+-------------------+------------------+
|                Food|     194812.0|             8387.0|              1588|
|Computers and ele...|     190692.5|             8272.0|              1558|
|          Patisserie|     182165.5|             7943.0|              1528|
|           Beverages|     197047.5|             8358.0|              1567|
|Electric househol...|     203813.5|             8309.0|              1591|
|       Milk Products|     180112.0|             8339.0|              1584|
|           Furniture|     195310.0|             8462.0|              1591|
|            Butchers|     208118.0|             8206.0|              1568|
+--------------------+-------------+-------------------+------------------+



In [76]:
#Top 10 customers by total spending

df_customers_by_total_spending = gold_delta_lake.groupBy("Customer_ID").agg(sum("Total_Spent").alias("Total_Spendings")
                                                                            ).orderBy(desc("Total_Spendings")).limit(10)
df_customers_by_total_spending.show()

+-----------+---------------+
|Customer_ID|Total_Spendings|
+-----------+---------------+
|    CUST_24|        68452.0|
|    CUST_08|        67351.5|
|    CUST_05|        66974.5|
|    CUST_16|        65570.5|
|    CUST_13|        65037.0|
|    CUST_23|        64507.0|
|    CUST_10|        63155.5|
|    CUST_15|        63117.5|
|    CUST_21|        62933.0|
|    CUST_02|        62046.5|
+-----------+---------------+



In [77]:
#Monthly Revenue Trend
from pyspark.sql.functions import month, year, monthname, sum

df_monthly_revenue = gold_delta_lake.groupBy(
    year('Transaction_Date').alias('Year'),
    monthname('Transaction_Date').alias('Month'),
    month('Transaction_Date').alias('Month_Num')
).agg(
    sum('Total_Spent').alias('Total_Revenue')
).orderBy('Year', 'Month_Num')

df_monthly_revenue.show()


+----+-----+---------+-------------+
|Year|Month|Month_Num|Total_Revenue|
+----+-----+---------+-------------+
|2022|  Jan|        1|      52911.5|
|2022|  Feb|        2|      43325.5|
|2022|  Mar|        3|      40996.0|
|2022|  Apr|        4|      40442.0|
|2022|  May|        5|      40347.5|
|2022|  Jun|        6|      42576.0|
|2022|  Jul|        7|      44471.5|
|2022|  Aug|        8|      41333.5|
|2022|  Sep|        9|      46113.5|
|2022|  Oct|       10|      38355.0|
|2022|  Nov|       11|      41256.5|
|2022|  Dec|       12|      38201.0|
|2023|  Jan|        1|      48052.5|
|2023|  Feb|        2|      39214.5|
|2023|  Mar|        3|      38534.5|
|2023|  Apr|        4|      38905.5|
|2023|  May|        5|      40480.5|
|2023|  Jun|        6|      42474.0|
|2023|  Jul|        7|      45632.5|
|2023|  Aug|        8|      38592.0|
+----+-----+---------+-------------+
only showing top 20 rows


In [78]:
#Online vs In-store sales comparison!

df_location_sales = gold_delta_lake.groupBy('Location').agg(sum('Total_Spent').alias('Total_Revenue'))

In [79]:
df_location_sales.show()

+--------+-------------+
|Location|Total_Revenue|
+--------+-------------+
|In-store|     760670.0|
|  Online|     791401.0|
+--------+-------------+



In [80]:
# Revenue with vs without discount!

df_discount_revenue = gold_delta_lake.groupBy('Discount_Applied').agg(sum('Total_Spent').alias('Total_Revenue'))

In [81]:
df_discount_revenue.show()

+----------------+-------------+
|Discount_Applied|Total_Revenue|
+----------------+-------------+
|           False|     515135.0|
|         Unknown|     512492.5|
|            True|     524443.5|
+----------------+-------------+



In [82]:
#Adding variable to .env file
gold_path = os.getenv("GOLD_PATH")

In [84]:
#Saving the aggregations as Delta tables

df_revenue_by_category.write.mode("overwrite").format("delta").save(gold_path + "\\revenue_by_category")
df_customers_by_total_spending.write.mode("overwrite").format("delta").save(gold_path + "\\top_customers")
df_monthly_revenue.write.mode("overwrite").format("delta").save(gold_path + "\\monthly_revenue")
df_location_sales.write.mode("overwrite").format("delta").save(gold_path + "\\location_sales")
df_discount_revenue.write.mode("overwrite").format("delta").save(gold_path + "\\discount_revenue")